In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
import os, json
import numpy as np
import torch

In [3]:
def load_jsonl(path):
    rows = []
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            rows.append(json.loads(line))
    return rows

In [5]:
train_big_path = "/content/drive/MyDrive/FYP/slsa/slsa_train_big.jsonl"
dev_path = "/content/drive/MyDrive/FYP/slsa/slsa_dev.jsonl"
test_path = "/content/drive/MyDrive/FYP/slsa/slsa_test.jsonl"

train_rows = load_jsonl(train_big_path)
dev_rows = load_jsonl(dev_path)
test_rows = load_jsonl(test_path)

In [6]:
LABELS = ["negative", "mixed", "positive"]
label2id = {lab:i for i, lab in enumerate(LABELS)}
id2label = {i:lab for lab, i in label2id.items()}

def normalize_label(l):
    l = l.lower()
    return l

In [7]:
from datasets import Dataset

def rows_to_dataset(rows):
    return Dataset.from_dict({
        "text":  [r["text"] for r in rows],
        "label": [label2id[normalize_label(r["label"])] for r in rows],
    })

train_ds = rows_to_dataset(train_rows)
dev_ds = rows_to_dataset(dev_rows)
test_ds = rows_to_dataset(test_rows)

In [8]:
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    DataCollatorWithPadding
)

model_name = "bert-base-multilingual-cased"

tokenizer = AutoTokenizer.from_pretrained(model_name)

def tokenize_fn(batch):
    return tokenizer(
        batch["text"],
        truncation=True,
        padding=False,
    )

train_ds = train_ds.map(tokenize_fn, batched=True)
dev_ds   = dev_ds.map(tokenize_fn, batched=True)
test_ds  = test_ds.map(tokenize_fn, batched=True)

config.json:   0%|          | 0.00/625 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/49.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/996k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.96M [00:00<?, ?B/s]

Map:   0%|          | 0/24000 [00:00<?, ? examples/s]

Map:   0%|          | 0/3000 [00:00<?, ? examples/s]

Map:   0%|          | 0/3001 [00:00<?, ? examples/s]

In [9]:
model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=len(LABELS),
    id2label=id2label,
    label2id=label2id,
)

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

model.safetensors:   0%|          | 0.00/714M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: bert-base-multilingual-cased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [10]:
from sklearn.metrics import f1_score, accuracy_score

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)

    acc = accuracy_score(labels, preds)
    macro = f1_score(labels, preds, average="macro")
    return {"acc": acc, "macro_f1": macro}

In [13]:
out_dir = "/content/drive/MyDrive/FYP/slsa/bert_mbert_slsa_big"

training_args = TrainingArguments(
    output_dir=out_dir,
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    num_train_epochs=3,
    weight_decay=0.01,
    logging_steps=50,
    load_best_model_at_end=True,
    metric_for_best_model="macro_f1",
    greater_is_better=True,
    fp16=torch.cuda.is_available(),
    report_to="none",
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=dev_ds,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

In [14]:
trainer.train()

Epoch,Training Loss,Validation Loss,Acc,Macro F1
1,0.704359,0.720861,0.668000,0.664730
2,0.615450,0.692511,0.686667,0.684104
3,0.529534,0.713682,0.697000,0.698867


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['bert.embeddings.LayerNorm.weight', 'bert.embeddings.LayerNorm.bias', 'bert.encoder.layer.0.attention.output.LayerNorm.weight', 'bert.encoder.layer.0.attention.output.LayerNorm.bias', 'bert.encoder.layer.0.output.LayerNorm.weight', 'bert.encoder.layer.0.output.LayerNorm.bias', 'bert.encoder.layer.1.attention.output.LayerNorm.weight', 'bert.encoder.layer.1.attention.output.LayerNorm.bias', 'bert.encoder.layer.1.output.LayerNorm.weight', 'bert.encoder.layer.1.output.LayerNorm.bias', 'bert.encoder.layer.2.attention.output.LayerNorm.weight', 'bert.encoder.layer.2.attention.output.LayerNorm.bias', 'bert.encoder.layer.2.output.LayerNorm.weight', 'bert.encoder.layer.2.output.LayerNorm.bias', 'bert.encoder.layer.3.attention.output.LayerNorm.weight', 'bert.encoder.layer.3.attention.output.LayerNorm.bias', 'bert.encoder.layer.3.output.LayerNorm.weight', 'bert.encoder.layer.3.output.LayerNorm.bias', 'bert.encoder.layer.4.attention.output.La

TrainOutput(global_step=4500, training_loss=0.6781612629360623, metrics={'train_runtime': 358.244, 'train_samples_per_second': 200.98, 'train_steps_per_second': 12.561, 'total_flos': 1.5270469648271712e+16, 'train_loss': 0.6781612629360623, 'epoch': 3.0})

In [15]:
test_metrics = trainer.evaluate(eval_dataset=test_ds)

In [16]:
trainer.save_model(out_dir)
tokenizer.save_pretrained(out_dir)

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

('/content/drive/MyDrive/FYP/slsa/bert_mbert_slsa_big/tokenizer_config.json',
 '/content/drive/MyDrive/FYP/slsa/bert_mbert_slsa_big/tokenizer.json')

In [17]:
result = {
    "acc": test_metrics["eval_acc"],
    "macro_f1": test_metrics["eval_macro_f1"],
    "n": len(test_rows)
}

In [18]:
print(result)

{'acc': 0.7057647450849717, 'macro_f1': 0.707564833744143, 'n': 3001}


In [19]:
save_path = "/content/drive/MyDrive/FYP/slsa/bert_results/mbert_slsa_big_results.json"
os.makedirs(os.path.dirname(save_path), exist_ok=True)

with open(save_path, "w", encoding="utf-8") as f:
    json.dump(result, f, ensure_ascii=False, indent=4)